In [1]:
from transformers import pipeline

model = pipeline(model="seara/rubert-tiny2-russian-sentiment")

W1119 18:22:01.395000 19196 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Device set to use cuda:0


In [2]:
from converter import OUTPUT_FILES as FILES
import re
import pandas as pd

def clean_text(text):
    if not isinstance(text, str):
        return text

    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\+?\d[\d\-\s\(\)]{7,}\d", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def split_to_chunks(text, max_words=400):
    words = text.split()
    chunks = []

    for i in range(0, len(words), max_words):
        chunks.append(" ".join(words[i:i + max_words]))

    return chunks

def classify_text_auto(text):
    text = clean_text(text)
    chunks = split_to_chunks(text)
    results = []

    for chunk in chunks:
        out = model(chunk)
        results.append(out[0]["label"])

    return max(set(results), key=results.count)

for csv in FILES:
    df = pd.read_csv(csv, sep=",", quotechar='"')
    df["ТОНАЛЬНОСТЬ"] = df["ТЕКСТ ПОСТА"].apply(classify_text_auto)
    new_name = f"{csv.replace('csv', 'out2').split('.')[0]}_С_Тональностью.csv"
    df.to_csv(new_name, index=False, encoding="utf-8-sig")
    print(f"Сохранен файл {new_name}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Сохранен файл data/out2/Санкт_Петербург_С_Тональностью.csv
